In [1]:
import import_ipynb
from Codigos import *
from Recodificacion import *

from pathlib import Path 
import json 
import joblib 
import numpy as np 
import pandas as pd 
import tensorflow as tf

In [2]:
# ============================================================
# CONFIGURACIÓN DE RUTAS
# ============================================================
from pathlib import Path

# Directorio de trabajo actual del notebook
BASE_DIR = Path.cwd()

DIR_ARTEFACTOS = BASE_DIR / "modelo_lpips_exportado"
DIR_RESULTADOS = BASE_DIR / "resultados_optimizacion"

RUTA_PREPROCESAMIENTO = (
    DIR_ARTEFACTOS / "preprocesamiento.joblib"
)
RUTA_MODELO = (
    DIR_RESULTADOS / "modelo_lpips_optimizado.keras"
)
RUTA_CONFIGURACION = (
    DIR_RESULTADOS / "configuracion_modelo.json"
)

print("Directorio de trabajo:", BASE_DIR)
print("Preprocesamiento:", RUTA_PREPROCESAMIENTO)
print("Modelo:", RUTA_MODELO)
print("Configuración:", RUTA_CONFIGURACION)

RUTA_PREPROCESAMIENTO = DIR_ARTEFACTOS / "preprocesamiento.joblib"
RUTA_MODELO = DIR_RESULTADOS / "modelo_lpips_optimizado.keras"
RUTA_CONFIGURACION = DIR_RESULTADOS / "configuracion_modelo.json"




Directorio de trabajo: d:\TrabajoGrado\Codigos\Funciones_Matematicas
Preprocesamiento: d:\TrabajoGrado\Codigos\Funciones_Matematicas\modelo_lpips_exportado\preprocesamiento.joblib
Modelo: d:\TrabajoGrado\Codigos\Funciones_Matematicas\resultados_optimizacion\modelo_lpips_optimizado.keras
Configuración: d:\TrabajoGrado\Codigos\Funciones_Matematicas\resultados_optimizacion\configuracion_modelo.json


In [3]:
def extraer_features(path, prefix):
    f = {}

    f[f"{prefix}_media_lum"] = obtener_media_luminancia(path)
    f[f"{prefix}_var_lum"] = obtener_varianza_luminancia(path)
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)


    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    f[f"{prefix}_grad_mean"] = obtener_contraste_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_glcm(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_glcm(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    f[f"{prefix}_entropia_shannon"] = obtener_entropia_shannon(path)
    f[f"{prefix}_entropia_espectral"] = obtener_entropia_espectral(path)
    f[f"{prefix}_entropia_gradiente"] = obtener_entropia_gradiente(path)
    f[f"{prefix}_entropia_glcm"] = obtener_entropia_glcm(path)

    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)
    f[f"{prefix}_hue_var"] = obtener_varianza_matiz(path)

    f[f"{prefix}_megapixeles"] = obtener_megapixeles(path)
    f[f"{prefix}_aspect_ratio"] = obtener_aspect_ratio(path)

    return f

In [4]:
# ============================================================
# CARGAR ARTEFACTOS UNA SOLA VEZ
# ============================================================

def cargar_artefactos():
    rutas = [
        RUTA_PREPROCESAMIENTO,
        RUTA_MODELO,
        RUTA_CONFIGURACION,
    ]

    faltantes = [str(ruta) for ruta in rutas if not ruta.exists()]

    if faltantes:
        raise FileNotFoundError(
            "No se encontraron los siguientes archivos:\n"
            + "\n".join(faltantes)
        )

    preprocesamiento = joblib.load(RUTA_PREPROCESAMIENTO)
    modelo = tf.keras.models.load_model(RUTA_MODELO)

    with open(RUTA_CONFIGURACION, "r", encoding="utf-8") as archivo:
        configuracion = json.load(archivo)

    return modelo, preprocesamiento, configuracion


MODELO, PREPROCESAMIENTO, CONFIGURACION = cargar_artefactos()

In [5]:
# ============================================================
# PREPARAR UNA IMAGEN PARA LA INFERENCIA
# ============================================================

def preparar_entrada(
    path_imagen,
    formato_salida,
    metodo,
    calidad,
):
    """
    Construye las variables de entrada usando las mismas reglas
    aplicadas durante la preparación del conjunto de entrenamiento.

    No recodifica la imagen: estima LPIPS antes de hacerlo.
    """

    path_imagen = Path(path_imagen)

    if not path_imagen.exists():
        raise FileNotFoundError(
            f"No se encontró la imagen: {path_imagen}"
        )

    if not path_imagen.is_file():
        raise ValueError("La ruta indicada no corresponde a un archivo.")

    # --------------------------------------------------------
    # 1. Detectar el formato original desde la extensión
    # --------------------------------------------------------

    formato_entrada = path_imagen.suffix.lower().lstrip(".")

    if formato_entrada == "jpeg":
        formato_entrada = "jpg"

    formatos_validos = {"jpg", "png", "webp"}

    if formato_entrada not in formatos_validos:
        raise ValueError(
            f"Formato de entrada no admitido: {formato_entrada}. "
            f"Formatos admitidos: {sorted(formatos_validos)}"
        )

    formato_salida = formato_salida.lower().strip().lstrip(".")

    if formato_salida == "jpeg":
        formato_salida = "jpg"

    if formato_salida not in formatos_validos:
        raise ValueError(
            f"Formato de salida no admitido: {formato_salida}"
        )

    if not isinstance(calidad, (int, np.integer)):
        raise TypeError("La calidad debe ser un número entero.")

    if calidad not in {10, 30, 50, 70, 90}:
        raise ValueError(
            "La calidad debe ser uno de los niveles "
            "experimentales: 10, 30, 50, 70 o 90."
        )

    # --------------------------------------------------------
    # 2. Extraer las características de la imagen original
    # --------------------------------------------------------

    caracteristicas = extraer_features(
        str(path_imagen),
        prefix="orig",
    )

    X = pd.DataFrame([caracteristicas])

    # --------------------------------------------------------
    # 3. Aplicar las mismas eliminaciones del entrenamiento
    # --------------------------------------------------------

    columnas_eliminar = [
        "orig_var_lum",
        "orig_edges_s1",
        "orig_edges_s2",
        "orig_hlfr",
        "orig_rango",
    ]

    X = X.drop(
        columns=columnas_eliminar,
        errors="ignore",
    )

    # --------------------------------------------------------
    # 4. Tratar hue_var como en el entrenamiento
    # --------------------------------------------------------

    if "orig_hue_var" not in X.columns:
        raise ValueError(
            "La extracción no generó la característica orig_hue_var."
        )

    hue_faltante = X["orig_hue_var"].isna().astype(int)

    X["orig_hue_var_faltante"] = hue_faltante

    mediana_hue = PREPROCESAMIENTO["mediana_hue"]

    X["orig_hue_var"] = X["orig_hue_var"].fillna(
        mediana_hue
    )

    # --------------------------------------------------------
    # 5. Añadir los parámetros de recodificación
    # --------------------------------------------------------

    X["formato_entrada"] = formato_entrada
    X["formato_salida"] = formato_salida
    X["metodo"] = metodo
    X["calidad"] = calidad

    columnas_categoricas = PREPROCESAMIENTO[
        "columnas_categoricas"
    ]

    columnas_modelo = PREPROCESAMIENTO["columnas_modelo"]

    # Verificar que las categorías seleccionadas existan
    # en el esquema usado durante el entrenamiento.
    for columna in columnas_categoricas:
        valor = X[columna].iloc[0]
        prefijo = f"{columna}_"

        categorias_conocidas = {
            nombre[len(prefijo):]
            for nombre in columnas_modelo
            if nombre.startswith(prefijo)
        }

        if valor not in categorias_conocidas:
            raise ValueError(
                f"Categoría no reconocida para '{columna}': "
                f"'{valor}'. Categorías disponibles: "
                f"{sorted(categorias_conocidas)}"
            )

    # --------------------------------------------------------
    # 6. Codificación One-Hot y alineación de columnas
    # --------------------------------------------------------

    X = pd.get_dummies(
        X,
        columns=columnas_categoricas,
        dtype=int,
    )

    # Reproduce el esquema de columnas del entrenamiento.
    # Las columnas ausentes se rellenan con cero.
    X = X.reindex(
        columns=columnas_modelo,
        fill_value=0,
    )

    # --------------------------------------------------------
    # 7. Normalizar exclusivamente variables continuas
    # --------------------------------------------------------

    columnas_continuas = PREPROCESAMIENTO[
        "columnas_continuas"
    ]

    X.loc[:, columnas_continuas] = (
        PREPROCESAMIENTO["scaler"].transform(
            X[columnas_continuas]
        )
    )

    # Comprobaciones finales
    if X.isna().any().any():
        columnas_nulas = X.columns[X.isna().any()].tolist()
        raise ValueError(
            f"Quedaron valores faltantes en la entrada: {columnas_nulas}"
        )

    if X.columns.tolist() != columnas_modelo:
        raise ValueError(
            "Las columnas no coinciden con las del entrenamiento."
        )

    return X

In [6]:
# ============================================================
# PREDICCIÓN ANTICIPADA DE LPIPS
# ============================================================

def predecir_lpips(
    path_imagen,
    formato_salida,
    metodo,
    calidad,
):
    """
    Estima LPIPS antes de realizar la recodificación.

    Devuelve tanto la predicción original del modelo como
    una estimación limitada inferiormente a cero para su
    interpretación dentro del dominio de LPIPS.
    """

    X = preparar_entrada(
        path_imagen=path_imagen,
        formato_salida=formato_salida,
        metodo=metodo,
        calidad=calidad,
    )

    prediccion_original = float(
        MODELO.predict(X, verbose=0).reshape(-1)[0]
    )

    prediccion_no_negativa = max(0.0, prediccion_original)

    return {
        "lpips_estimado_original": prediccion_original,
        "lpips_estimado_no_negativo": prediccion_no_negativa,
        "prediccion_negativa": prediccion_original < 0,
        "formato_entrada": Path(path_imagen).suffix.lower().lstrip("."),
        "formato_salida": formato_salida,
        "metodo": metodo,
        "calidad": calidad,
    }




In [7]:
# ============================================================
# EJEMPLO DE USO
# ============================================================

resultado = predecir_lpips(
        path_imagen="Imagenes/jpg/apparel (1) (Copy).jpg",
        formato_salida="webp",
        metodo="pillow",
        calidad=50,
    )

print("\nRESULTADO DE LA PREDICCIÓN")
print("=" * 45)

for clave, valor in resultado.items():
    print(f"{clave}: {valor}")


RESULTADO DE LA PREDICCIÓN
lpips_estimado_original: 0.02168445661664009
lpips_estimado_no_negativo: 0.02168445661664009
prediccion_negativa: False
formato_entrada: jpg
formato_salida: webp
metodo: pillow
calidad: 50


C:\Users\santi\AppData\Local\Temp\ipykernel_15208\163566665.py:170: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.00940817]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[:, columnas_continuas] = (


In [8]:
from PIL import Image

def verificar_rgb(ruta_imagen: str) -> bool:
    """
    Verifica si una imagen está en modo RGB.

    Retorna:
        True: si la imagen está en modo RGB.
        False: si está en otro modo, como RGBA, L, P o CMYK.
    """
    with Image.open(ruta_imagen) as imagen:
        return imagen.mode == "RGB"

In [9]:
print(verificar_rgb("Imagenes/jpg/apparel (1) (Copy).jpg"))

True


In [10]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io
from skimage.util import img_as_ubyte
import lpips

# ---------------------------
# CONFIGURACIÓN
# ---------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn_lpips = lpips.LPIPS(net='alex').to(device)  # red elegida: AlexNet (mejor correlación en BAPPS, Zhang et al., 2018)

# ---------------------------
# VALIDACIÓN DE FORMATO (sin conversión)
# ---------------------------

def es_rgb_valida(img):
    """
    Verifica si una imagen ya está en formato RGB de 3 canales,
    sin realizar ninguna conversión.

    Retorna
    -------
    bool
        True si la imagen es RGB (3 canales), False en cualquier
        otro caso (escala de grises, RGBA, u otro formato).
    """
    return img.ndim == 3 and img.shape[-1] == 3

def load_tensor(path, device):
    """
    Carga una imagen y la convierte a tensor solo si ya está en
    formato RGB. Si no lo está (escala de grises, RGBA, etc.),
    retorna None para que sea omitida.

    Retorna
    -------
    torch.Tensor o None
        Tensor listo para LPIPS, o None si la imagen debe omitirse.
    """
    img = io.imread(path)

    if not es_rgb_valida(img):
        return None

    img = img_as_ubyte(img)
    img = img.astype(np.float32) / 127.5 - 1.0
    return torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = F.interpolate(
            img2,
            size=img1.shape[2:],
            mode='bilinear',
            align_corners=False
        )
    return img1, img2

# ---------------------------
# LPIPS
# ---------------------------

def calcular_lpips(img_orig_path, img_comp_path):
    """
    Calcula la distancia perceptual LPIPS entre dos imágenes.

    Si alguna de las dos imágenes no está en formato RGB (por
    ejemplo, escala de grises o RGBA), la comparación se omite
    y se retorna None en lugar de forzar una conversión.

    Parámetros
    ----------
    img_orig_path : str
        Ruta de la imagen original.
    img_comp_path : str
        Ruta de la imagen comprimida o transformada.

    Retorna
    -------
    float o None
        Distancia perceptual LPIPS, o None si alguna imagen
        fue omitida por no ser RGB.
    """
    orig = load_tensor(img_orig_path, device)
    comp = load_tensor(img_comp_path, device)

    if orig is None or comp is None:
        return None

    orig, comp = match_size(orig, comp)

    with torch.no_grad():
        dist = loss_fn_lpips(orig, comp)

    return dist.item()

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\lpips\weights\v0.1\alex.pth


In [11]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

DIRECTORIO_RECODIFICADAS = Path("Imagenes_prueba_recodificadas")

EXTENSIONES = {
    "jpg": ".jpg",
    "png": ".png",
    "webp": ".webp",
}

COLUMNAS_RESULTADOS = [
    "nombre_imagen",
    "calidad",
    "formato_entrada",
    "formato_salida",
    "metodo",
    "lpips_estimado_original",
    "lpips_estimado_no_negativo",
    "prediccion_negativa",
    "lpips_real",
    "ruta_imagen_recodificada",
]


In [12]:
def recodificar_y_calcular_lpips(
    path_imagen,
    formato_salida,
    calidad,
    metodo,
    directorio_salida=DIRECTORIO_RECODIFICADAS,
):
    """
    Recodifica la imagen mediante FFmpeg, ImageMagick o Pillow
    y calcula el LPIPS real entre el original y el resultado.

    Retorna:
        {
            "lpips_real": float,
            "ruta_imagen_recodificada": str
        }
    """

    path_imagen = Path(path_imagen)

    if not path_imagen.is_file():
        raise FileNotFoundError(
            f"No existe la imagen original: {path_imagen}"
        )

    formato_salida = formato_salida.lower().strip().lstrip(".")

    if formato_salida == "jpeg":
        formato_salida = "jpg"

    if formato_salida not in EXTENSIONES:
        raise ValueError(
            f"Formato de salida no válido: {formato_salida}. "
            f"Formatos permitidos: {list(EXTENSIONES)}"
        )

    metodo_normalizado = metodo.lower().strip()

    alias_metodos = {
        "ffmpeg": "ffmpeg",
        "imagemagick": "imagemagick",
        "image magick": "imagemagick",
        "image_magick": "imagemagick",
        "pillow": "pillow",
    }

    if metodo_normalizado not in alias_metodos:
        raise ValueError(
            f"Método no válido: {metodo}. "
            "Utiliza FFmpeg, ImageMagick o Pillow."
        )

    metodo_normalizado = alias_metodos[metodo_normalizado]

    # --------------------------------------------------------
    # Preparar la ruta del archivo recodificado
    # --------------------------------------------------------

    directorio_salida = Path(directorio_salida)

    carpeta_salida = (
        directorio_salida
        / metodo_normalizado
        / formato_salida
    )

    carpeta_salida.mkdir(parents=True, exist_ok=True)

    nombre_salida = (
        f"{path_imagen.stem}"
        f"_q{calidad}"
        f"{EXTENSIONES[formato_salida]}"
    )

    path_recodificada = carpeta_salida / nombre_salida

    # Eliminar una salida anterior con el mismo nombre
    # para evitar utilizar accidentalmente un resultado obsoleto.
    if path_recodificada.exists():
        path_recodificada.unlink()

    # --------------------------------------------------------
    # Ejecutar el método de recodificación seleccionado
    # --------------------------------------------------------

    if metodo_normalizado == "ffmpeg":

        convert_with_ffmpeg(
            str(path_imagen),
            str(path_recodificada),
            quality=calidad,
        )

    elif metodo_normalizado == "imagemagick":

        convert_with_imagemagick(
            str(path_imagen),
            str(path_recodificada),
            quality=calidad,
        )

    elif metodo_normalizado == "pillow":

        convert_image_pillow(
            str(path_imagen),
            str(path_recodificada),
            quality=calidad,
        )

    # --------------------------------------------------------
    # Verificar que la recodificación generó el archivo
    # --------------------------------------------------------

    if not path_recodificada.is_file():
        raise RuntimeError(
            f"El método {metodo_normalizado} no generó "
            f"la imagen esperada: {path_recodificada}"
        )

    # --------------------------------------------------------
    # Calcular LPIPS real entre el original y el recodificado
    # --------------------------------------------------------

    lpips_real = float(
        calcular_lpips(
            str(path_imagen),
            str(path_recodificada),
        )
    )

    return {
        "lpips_real": lpips_real,
        "ruta_imagen_recodificada": str(path_recodificada),
    }

In [13]:
# ============================================================
# 2. PREDECIR, RECODIFICAR Y REGISTRAR EN EL DATAFRAME
# ============================================================

def evaluar_imagen_y_registrar(
    path_imagen,
    formato_salida,
    df,
    calidad,
    metodo,
):
    """
    Flujo completo:

    1. Predice LPIPS antes de recodificar.
    2. Recodifica la imagen con el método seleccionado.
    3. Calcula el LPIPS real del par.
    4. Guarda ambas mediciones en el DataFrame.

    Retorna:
        df_actualizado, resultado
    """

    path_imagen = Path(path_imagen)

    if not path_imagen.is_file():
        raise FileNotFoundError(
            f"No existe la imagen: {path_imagen}"
        )

    if df is None:
        df = pd.DataFrame()

    # --------------------------------------------------------
    # PASO 1. Predicción anticipada antes de recodificar
    # --------------------------------------------------------

    prediccion = predecir_lpips(
        path_imagen=str(path_imagen),
        formato_salida=formato_salida,
        metodo=metodo,
        calidad=calidad,
    )

    # --------------------------------------------------------
    # PASO 2. Recodificación y LPIPS real
    # --------------------------------------------------------

    medicion_real = recodificar_y_calcular_lpips(
        path_imagen=str(path_imagen),
        formato_salida=formato_salida,
        calidad=calidad,
        metodo=metodo,
    )

    # --------------------------------------------------------
    # PASO 3. Construir el registro de resultados
    # --------------------------------------------------------

    resultado = {
        "nombre_imagen": path_imagen.name,
        "calidad": calidad,
        "formato_entrada": prediccion["formato_entrada"],
        "formato_salida": prediccion["formato_salida"],
        "metodo": prediccion["metodo"],
        "lpips_estimado_original": float(
            prediccion["lpips_estimado_original"]
        ),
        "lpips_estimado_no_negativo": float(
            prediccion["lpips_estimado_no_negativo"]
        ),
        "prediccion_negativa": bool(
            prediccion["prediccion_negativa"]
        ),
        "lpips_real": medicion_real["lpips_real"],
        "ruta_imagen_recodificada": (
            medicion_real["ruta_imagen_recodificada"]
        ),
    }

    # --------------------------------------------------------
    # PASO 4. Incorporar el registro al DataFrame
    # --------------------------------------------------------

    fila = pd.DataFrame([resultado], columns=COLUMNAS_RESULTADOS)

    # Conserva las columnas que ya tuviera el DataFrame.
    df_actualizado = pd.concat(
        [df, fila],
        ignore_index=True,
    )

    return df_actualizado, resultado

In [14]:
import random
import pandas as pd
from pathlib import Path


# ============================================================
# PARÁMETROS DE LA EVALUACIÓN
# ============================================================

METODOS = ["pillow", "ffmpeg", "imagemagick"]
FORMATOS = ["jpg", "webp", "png"]
CALIDADES_POOL = [10, 30, 50, 70, 90]

CARPETA_PRUEBA = "Imagenes_prueba"

ARCHIVO_RESULTADOS = "resultados_prueba_modelo.csv"
ARCHIVO_ERRORES = "errores_prueba_modelo.csv"

EXTENSIONES_VALIDAS = {".jpg", ".jpeg", ".png", ".webp"}


# ============================================================
# EVALUAR TODAS LAS IMÁGENES CON PARÁMETROS ALEATORIOS
# ============================================================

def evaluar_dataset_prueba(
    df_resultados=None,
    carpeta_prueba=CARPETA_PRUEBA,
    seed=42,
):
    """
    Recorre las imágenes de Imagenes_prueba y realiza un
    proceso aleatorio por imagen.

    Para cada imagen:
        1. Identifica su formato de entrada.
        2. Selecciona un método aleatorio.
        3. Selecciona un formato de salida diferente al de entrada.
        4. Selecciona una calidad aleatoria.
        5. Predice LPIPS antes de recodificar.
        6. Recodifica y calcula el LPIPS real.
        7. Registra los resultados en un DataFrame y un CSV.

    Requiere la función previamente definida:
        evaluar_imagen_y_registrar()

    Retorna:
        df_resultados, df_errores
    """

    rng = random.Random(seed)
    carpeta_prueba = Path(carpeta_prueba)

    if not carpeta_prueba.is_dir():
        raise FileNotFoundError(
            f"No existe la carpeta de prueba: "
            f"{carpeta_prueba.resolve()}"
        )

    # --------------------------------------------------------
    # 1. Encontrar todas las imágenes, incluidas las
    #    ubicadas en Imagenes_prueba/jpg, png y webp
    # --------------------------------------------------------

    imagenes = sorted(
        ruta
        for ruta in carpeta_prueba.rglob("*")
        if ruta.is_file()
        and ruta.suffix.lower() in EXTENSIONES_VALIDAS
    )

    if not imagenes:
        raise ValueError(
            f"No se encontraron imágenes en {carpeta_prueba.resolve()}"
        )

    columnas = [
        "nombre_imagen",
        "ruta_imagen_original",
        "calidad",
        "formato_entrada",
        "formato_salida",
        "metodo",
        "lpips_estimado_original",
        "lpips_estimado_no_negativo",
        "prediccion_negativa",
        "lpips_real",
        "ruta_imagen_recodificada",
    ]

    if df_resultados is None:
        df_resultados = pd.DataFrame(columns=columnas)

    errores = []
    total = len(imagenes)

    print("=" * 75)
    print("EVALUACIÓN PRÁCTICA DEL MODELO PREDICTIVO DE LPIPS")
    print("=" * 75)
    print(f"Carpeta de prueba: {carpeta_prueba.resolve()}")
    print(f"Imágenes encontradas: {total}")
    print("Se aplicará un proceso aleatorio a cada imagen.\n")

    # --------------------------------------------------------
    # 2. Procesar las imágenes una por una
    # --------------------------------------------------------

    for i, ruta_imagen in enumerate(imagenes, start=1):

        # Normalizar el formato de entrada.
        formato_entrada = ruta_imagen.suffix.lower().lstrip(".")

        if formato_entrada == "jpeg":
            formato_entrada = "jpg"

        # Seleccionar método y calidad aleatoriamente.
        metodo = rng.choice(METODOS)
        calidad = rng.choice(CALIDADES_POOL)

        # IMPORTANTE:
        # El formato de salida debe ser diferente al de entrada.
        formatos_disponibles = [
            formato
            for formato in FORMATOS
            if formato != formato_entrada
        ]

        if not formatos_disponibles:
            raise ValueError(
                f"No hay formatos de salida válidos para "
                f"{formato_entrada.upper()}."
            )

        formato_salida = rng.choice(formatos_disponibles)

        print("\n" + "-" * 75)
        print(f"Imagen {i}/{total}: {ruta_imagen.name}")
        print(f"Ruta original: {ruta_imagen}")
        print(f"Formato de entrada: {formato_entrada}")
        print(f"Método seleccionado: {metodo}")
        print(f"Formato de salida: {formato_salida}")
        print(f"Calidad: {calidad}")

        # ----------------------------------------------------
        # 3. Predecir LPIPS, recodificar y calcular LPIPS real
        # ----------------------------------------------------

        try:
            df_resultados, resultado = evaluar_imagen_y_registrar(
                path_imagen=str(ruta_imagen),
                formato_salida=formato_salida,
                df=df_resultados,
                calidad=calidad,
                metodo=metodo,
            )

            # Añadir la ruta exacta de la imagen original.
            df_resultados.loc[
                df_resultados.index[-1],
                "ruta_imagen_original",
            ] = str(ruta_imagen.resolve())

            print(
                f"LPIPS estimado original: "
                f"{resultado['lpips_estimado_original']:.6f}"
            )
            print(
                f"LPIPS estimado no negativo: "
                f"{resultado['lpips_estimado_no_negativo']:.6f}"
            )
            print(
                f"LPIPS real: "
                f"{resultado['lpips_real']:.6f}"
            )
            print(
                f"Predicción negativa: "
                f"{resultado['prediccion_negativa']}"
            )

            # Guardar el progreso después de cada imagen.
            df_resultados.to_csv(
                ARCHIVO_RESULTADOS,
                index=False,
                encoding="utf-8-sig",
            )

        except Exception as error:
            print(
                f"[ERROR] No se pudo procesar "
                f"{ruta_imagen.name}: {error}"
            )

            errores.append({
                "nombre_imagen": ruta_imagen.name,
                "ruta_imagen_original": str(ruta_imagen.resolve()),
                "formato_entrada": formato_entrada,
                "formato_salida": formato_salida,
                "metodo": metodo,
                "calidad": calidad,
                "error": str(error),
            })

            # Registrar errores sin interrumpir las imágenes restantes.
            pd.DataFrame(errores).to_csv(
                ARCHIVO_ERRORES,
                index=False,
                encoding="utf-8-sig",
            )

    # --------------------------------------------------------
    # 4. Guardar el resumen final
    # --------------------------------------------------------

    df_errores = pd.DataFrame(errores)

    df_resultados.to_csv(
        ARCHIVO_RESULTADOS,
        index=False,
        encoding="utf-8-sig",
    )

    print("\n" + "=" * 75)
    print("EVALUACIÓN FINALIZADA")
    print("=" * 75)
    print(f"Imágenes encontradas: {total}")
    print(f"Procesadas correctamente: {total - len(errores)}")
    print(f"Errores: {len(errores)}")
    print(f"Resultados guardados en: {Path(ARCHIVO_RESULTADOS).resolve()}")

    if errores:
        print(
            f"Errores guardados en: "
            f"{Path(ARCHIVO_ERRORES).resolve()}"
        )

    return df_resultados, df_errores


# ============================================================
# EJECUTAR LA EVALUACIÓN
# ============================================================

df_resultados, df_errores = evaluar_dataset_prueba(
    carpeta_prueba="Imagenes_prueba",
    seed=42,
)

EVALUACIÓN PRÁCTICA DEL MODELO PREDICTIVO DE LPIPS
Carpeta de prueba: D:\TrabajoGrado\Codigos\Funciones_Matematicas\Imagenes_prueba
Imágenes encontradas: 120
Se aplicará un proceso aleatorio a cada imagen.


---------------------------------------------------------------------------
Imagen 1/120: COCO_test2014_000000000862.jpg
Ruta original: Imagenes_prueba\jpg\COCO_test2014_000000000862.jpg
Formato de entrada: jpg
Método seleccionado: imagemagick
Formato de salida: webp
Calidad: 10


C:\Users\santi\AppData\Local\Temp\ipykernel_15208\163566665.py:170: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.42501692]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[:, columnas_continuas] = (


[ImageMagick] Convertido → Imagenes_prueba_recodificadas\imagemagick\webp\COCO_test2014_000000000862_q10.webp


C:\Users\santi\AppData\Local\Temp\ipykernel_15208\244610797.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_actualizado = pd.concat(


LPIPS estimado original: 0.339245
LPIPS estimado no negativo: 0.339245
LPIPS real: 0.305163
Predicción negativa: False

---------------------------------------------------------------------------
Imagen 2/120: COCO_test2014_000000020048.jpg
Ruta original: Imagenes_prueba\jpg\COCO_test2014_000000020048.jpg
Formato de entrada: jpg
Método seleccionado: imagemagick
Formato de salida: webp
Calidad: 50


...

C:\Users\santi\AppData\Local\Temp\ipykernel_15208\163566665.py:170: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.00940817]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[:, columnas_continuas] = (


[ImageMagick] Convertido → Imagenes_prueba_recodificadas\imagemagick\jpg\t10f_20260130003745_000982_d26a8080_q50.jpg
LPIPS estimado original: 0.042146
LPIPS estimado no negativo: 0.042146
LPIPS real: 0.044559
Predicción negativa: False

---------------------------------------------------------------------------
Imagen 119/120: t10f_20260130005002_000247_de26d343.webp
Ruta original: Imagenes_prueba\webp\t10f_20260130005002_000247_de26d343.webp
Formato de entrada: webp
Método seleccionado: imagemagick
Formato de salida: png
Calidad: 50


C:\Users\santi\AppData\Local\Temp\ipykernel_15208\163566665.py:170: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.00940817]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[:, columnas_continuas] = (


[ImageMagick] Convertido → Imagenes_prueba_recodificadas\imagemagick\png\t10f_20260130005002_000247_de26d343_q50.png
LPIPS estimado original: 0.000000
LPIPS estimado no negativo: 0.000000
LPIPS real: 0.000000
Predicción negativa: False

---------------------------------------------------------------------------
Imagen 120/120: t10f_20260130005002_000942_87e870b2.webp
Ruta original: Imagenes_prueba\webp\t10f_20260130005002_000942_87e870b2.webp
Formato de entrada: webp
Método seleccionado: ffmpeg
Formato de salida: jpg
Calidad: 10
[FFmpeg] Convertido → Imagenes_prueba_recodificadas\ffmpeg\jpg\t10f_20260130005002_000942_87e870b2_q10.jpg
LPIPS estimado original: 0.190175
LPIPS estimado no negativo: 0.190175
LPIPS real: 0.199636
Predicción negativa: False

EVALUACIÓN FINALIZADA
Imágenes encontradas: 120
Procesadas correctamente: 120
Errores: 0
Resultados guardados en: D:\TrabajoGrado\Codigos\Funciones_Matematicas\resultados_prueba_modelo.csv


C:\Users\santi\AppData\Local\Temp\ipykernel_15208\163566665.py:170: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.42501692]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X.loc[:, columnas_continuas] = (
